In [1]:
%pip install openai python-dotenv

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import requests

/Users/muhammadali/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
load_dotenv()
client = OpenAI()

In [4]:
def get_weather(city):
    url = f"https://wttr.in/{city}?format=j1"
    response = requests.get(url, timeout=10)
    if response.status_code == 200:
        return f"The weather in {city} is {response.text}"


In [5]:
available_tools = {
    "get_weather": get_weather
}

In [6]:
system_prompt = """Hey you are an expert AI assistant,
in resolving user queries using chain of thought.
You work on START, PLAN and OUTPUT steps.
You need to first plan what need to be done. The plan can be multiple steps. After planning, you need to provide the output.
Once you think enough plan has been done, finally you can give an OUTPUT.
you can also call a tool if required, from a list of available tools
for every tool call wait for the observe step which is the output from the called tool.

RULES:
- Strictly follow the given JSON output format
- Only run one step at a time. Do not run multiple steps in one go.
- The sequence of step is 
START (Where user gives an input query)
PLAN (That can be multiple times/steps)
OUTPUT (Which is going to be displayed to the user)

Available Tools:
- get_weather(city: str): Takes city name as an input and return weather information as Temprature(C): xC and Feels Like: yC

OUTPUT JSON FORMAT:
{"step": "start" | "plan" | "output" | "tool", "content": "string", "tool": "string", "input": "string"}

Example 1:
START: Can you solve 2 + 5 * 5/ 5
PLAN: {"step": "plan", "content": "Seems like user is interested in maths problem."}
PLAN: {"step": "plan", "content": "Looking at problem we should solve using BODMAS rule.:}
PLAN: {"step": "plan", "content": "First we need to solve 5 * 5 = 25"}
PLAN: {"step": "plan", "content": "Then we need to solve 25 / 5 = 5"}
PLAN: {"step": "plan", "content": "Finally we need to solve 2 + 5 = 7"}
OUTPUT: {"step": "output", "content": "7"}

Example 2: 
START: can you tell me the current weather of Koblenz
PLAN: {"step": "plan", "content": "Seems like user wants to know the weather of koblenz."}
PLAN: {"step": "plan", "content": "Lets see if we have any related tool from the list of available tools."}
PLAN: {"step": "plan", "content": "Great, we have a tool available for this user query."}
PLAN: {"step": "plan", "content": "I need to call get_weather tool for this query"}
PLAN: {"step": "tool", "tool": "get_weather", "content": "Koblenz"}
PLAN: {"step": "observe", "tool": "get_weather", "output": "The temprature of Koblenz is cloudy, 15C and feels like 10C"}
PLAN: {"step": "plan", "content": "I have got the current weather of Koblenz."}
OUTPUT: {"step": "output", "content": "The current weather in Koblenz is 15C with some clouds and feel like 10C"}
"""

In [7]:

import json

message_history = [
    {"role": "system", "content": system_prompt}
]

user_query = input("User Input: ")
message_history.append({"role": "user", "content": user_query})

while True:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={"type": "json_object"},
        messages=message_history
    )

    raw_result = response.choices[0].message.content

     # Save the assistant's response in conversation history
    message_history.append({
            "role": "assistant",
            "content": raw_result
        })
    
    # Convert JSON string into a Python dictionary
    parsed_result = json.loads(raw_result)


    step = parsed_result.get("step", "").lower()
    content = parsed_result.get("content", "")

    if step == "start":
        print("Start:", content)
        continue

    elif step == "plan":
        print("Plan:", content)
        continue

    elif step == "tool":
            tool_to_call = parsed_result.get("tool")
            tool_input = parsed_result.get("input")
            print(f"Tool: {tool_to_call}({tool_input})")
            tool_response = available_tools[tool_to_call](tool_input)
            message_history.append({"role": "developer", "content": json.dumps(
                 {"step": "OBSERVE", "tool": tool_to_call, "input": tool_input, "output": tool_response}
            )})
            continue

    elif step == "output":
        print("Output:", content)
        break

    else:
        print("Unknown step:", step)
        break


Start: weather of koblenz right now
Plan: Seems like user wants to know the current weather of Koblenz.
Plan: Let's see if we have any related tool from the list of available tools.
Plan: Great, we have a tool available for this user query.
Plan: I need to call get_weather tool for this query.
Tool: get_weather(None)
Plan: I have received the current weather data for Koblenz. It shows that the temperature is 12C, feels like 11C, and the weather is described as 'Patchy rain nearby'.
Plan: Now, let's summarize the weather information to create an output for the user.
Output: The current weather in Koblenz is 12C with patchy rain nearby, and it feels like 11C.
